In [78]:
import numpy as np
import pandas as pd

seed= 42

In [79]:
from gensim.models import Word2Vec

train_df= pd.read_csv('train_data.csv')
test_df= pd.read_csv('test_data.csv')
unlabeled_df= pd.read_csv('unlabeled_data.csv')

vocab= set()
for text in unlabeled_df['text']:
    vocab.update(text.split())

vocab_size= len(vocab)

sentences= [sentence.split() for sentence in unlabeled_df['text']]

word_vec= Word2Vec(
    sentences,
    vector_size=32,
    epochs=100,
    seed= seed
)

In [80]:
# Subtask 1
answer1= pd.DataFrame([{
    'subtaskID': 1,
    'datapointID': 0,
    'answer': vocab_size
}])

In [81]:
# Subtask 2

from sklearn.preprocessing import normalize

def embed_sentence(text):
    vectors= [
        word_vec.wv[w]
        for w in text.split()
        if w in word_vec.wv
    ]
    
    if len(vectors)== 0:
        return np.zeros(word_vec.vector_size)
    return np.mean(vectors, axis=0)

X_train= normalize(np.vstack([embed_sentence(sentence) for sentence in train_df['text']]))
y_train= train_df['label'].values

X_test= normalize(np.vstack([embed_sentence(sentence) for sentence in test_df['text']]))

In [82]:
from sklearn.linear_model import LogisticRegressionCV

model = LogisticRegressionCV(
    cv=5,
    max_iter=3000,
    random_state=seed
)

model.fit(X_train, y_train)

predictions= model.predict(X_test)

answer2= pd.DataFrame([{
    'subtaskID': 2,
    'datapointID': id_,
    'answer': int(pred)
} for id_, pred in zip(test_df['id'], predictions)])

In [83]:
answer= pd.concat([answer1, answer2])
answer.to_csv("submission.csv", index= False)